In [ ]:
from pathlib import Path
import subprocess, sys, json, zipfile
work=Path('/kaggle/working')
script=work/'step03c_preflight.py'
script.write_text('"""Bounded artifact and runtime preflight for the published ns64 ETs pilot."""\nimport hashlib\nimport importlib.metadata\nimport json\nimport platform\nimport subprocess\nimport time\nimport zipfile\nfrom pathlib import Path\n\n\ndef digest(path):\n    value = hashlib.sha256()\n    with path.open(\'rb\') as handle:\n        for block in iter(lambda: handle.read(1048576), b\'\'):\n            value.update(block)\n    return value.hexdigest()\n\n\ndef run(plan, input_root, output):\n    started = time.monotonic()\n    report = dict(team=plan[\'team\'], dataset=\'ETs\', status=\'BLOCKED\',\n                  hardware={}, checkpoints={}, stages={}, predictions=None, scores=None)\n    stage = \'hardware\'\n    try:\n        gpu = subprocess.check_output(\n            [\'nvidia-smi\', \'--query-gpu=name,memory.total,driver_version\', \'--format=csv,noheader\'],\n            timeout=10, text=True)\n        report[\'hardware\'] = dict(gpus=gpu.splitlines(), python=platform.python_version())\n        packages = {}\n        for name in [\'torch\', \'pycolmap\', \'kornia\', \'asmk\', \'curope\', \'mast3r\']:\n            try:\n                packages[name] = importlib.metadata.version(name)\n            except importlib.metadata.PackageNotFoundError:\n                packages[name] = None\n        report[\'packages\'] = packages\n        probe = subprocess.run([__import__(\'sys\').executable, \'-c\',\n            \'import torch; print(torch.ones(1, device="cuda").item()); import pycolmap, asmk, curope, mast3r\'],\n            capture_output=True, text=True, timeout=90)\n        report[\'import_cuda_probe\'] = dict(returncode=probe.returncode, stderr=probe.stderr[-2000:])\n        report[\'stages\'][stage] = \'PASS\'\n        stage = \'source\'\n        source = next(input_root.rglob(\'ns64_pilot_source.zip.bin\'), None)\n        if source is None or digest(source) != plan[\'source_zip_sha256\']:\n            raise ValueError(\'Mount the exact ns64_pilot_source.zip.bin from the preparation bundle\')\n        report[\'source_sha256\'] = digest(source)\n        report[\'stages\'][stage] = \'PASS\'\n        stage = \'inputs\'\n        candidates = [p for p in input_root.rglob(\'train/ETs\') if p.is_dir()]\n        if len(candidates) != 1:\n            raise ValueError(\'Expected one competition train/ETs directory\')\n        for name, expected in plan[\'images\'].items():\n            if digest(candidates[0] / name) != expected:\n                raise ValueError(\'ETs image identity mismatch\')\n        report[\'stages\'][stage] = \'PASS\'\n        stage = \'checkpoints\'\n        for key, filename in plan[\'weight_files\'].items():\n            paths = list(input_root.rglob(filename))\n            report[\'checkpoints\'][key] = [{\'path\':str(p), \'sha256\':digest(p)} for p in paths if p.is_file()]\n        raise RuntimeError(\'Original checkpoint versions and complete compatible environment are not pinned. Review the inventory before inference.\')\n    except Exception as exc:\n        report.update(failed_stage=stage, error=str(exc))\n        report[\'stages\'][stage] = \'BLOCKED\'\n    report[\'elapsed_seconds\'] = time.monotonic() - started\n    with zipfile.ZipFile(output, \'w\', zipfile.ZIP_DEFLATED) as archive:\n        for name, data in [(\'plan\', plan), (\'result\', report)]:\n            archive.writestr(name+\'.json\', json.dumps(data, indent=2))\n    return report\n\nplan={\'team\': \'ns64\', \'revision\': \'07b5695b99a8ae4725d5d952f0065c2157fadd7d\', \'config\': \'conf/pipeline/imc2025/mast3rhybrid/mast3rhybrid-022-c.yaml\', \'config_sha256\': \'e7829be60865192d3ddcc8555b94bc8678415e265a566d8ce8386ef17b0cc763\', \'source_zip_sha256\': \'c6035c596a484331ebfaf8d3b6368b4a32acb00b3bebe670a8215c78b6c12549\', \'budget_seconds\': 1800, \'images\': {\'et_et005.png\': \'675160dd2144ae9b8ad1ae30f4e86089b44bc9be34148a0d3ac3215b051af0e9\', \'et_et003.png\': \'9b2db887af9398e00324cf914805cb3cb089c4e23fe093d59f8f4d9601111ff9\', \'outliers_out_et002.png\': \'c23234d646b7c9f91183fde1731e38eff36a09397fae0e8e33790650d0b62522\', \'outliers_out_et001.png\': \'756f0702e44274fd9ac138bc94f53e06bd9664d2c34b527ceddafef8fd33b7b8\', \'et_et008.png\': \'12f60ddc5c989e47ef28ab42159e165560c9b62ed99775978544af428e4263e8\', \'another_et_another_et010.png\': \'1107c88723355ccecd548c14c0cac8d963866e551fb64234547de7fcfe64ba4d\', \'another_et_another_et007.png\': \'29d905cf262e5b40a6ef84e066a0aa47badcd81e5483991a204294a40c3d6dfc\', \'et_et007.png\': \'7abaa1d4666fd6f22d7eab5ac0066899410890ee7337a50e0d93b4735e9f232c\', \'another_et_another_et005.png\': \'f5651313171525bcd1ab24834f21d8fcbe1fb0b0ec89ce073442b96fc07158cf\', \'et_et006.png\': \'30b16b76ef40c0da407b3f295fa7e6f58cfa31416c3f5efc1ad219b0ed7618df\', \'another_et_another_et008.png\': \'602b33fdac149024b2c6d9cf011c818707f9a2b0e985e79f4dbf1adb6c55c4d1\', \'another_et_another_et001.png\': \'d126237e3c25c89cc876f660f80e357f07c85d3e0619e673755215df2dd576c8\', \'another_et_another_et002.png\': \'1cdf38c68f8d64900ea0c853621f32701ee98bc0257d6ff9e4b7938074c2cbcc\', \'et_et002.png\': \'7d746f4c3c26d56d74a895c6bed41d83b4de23135a9649461f25d4585b14a481\', \'another_et_another_et003.png\': \'6fc638634eaea0cbe708b9f330f00ce7be89456011b46037e96afd49d7991698\', \'another_et_another_et004.png\': \'ca44ba215925500b29b62e7e62c9b84a500b272c9b2f8de2d5bf45531f2c8373\', \'et_et000.png\': \'4a0c394180905103a0f090eb37de471a8d4949a91a3bf2950db4d011003e5b7b\', \'outliers_out_et003.png\': \'41d1e25d715765cefd0b7664156a3bebc2407587e7877c3ed9d04b1367d3253b\', \'another_et_another_et006.png\': \'3fdb25d9e49a253c3cf05b3010f19bee3cae8d8e4b76bea15f1be922aacdf367\', \'et_et001.png\': \'61c8043bd5caaaf9ac88a95665528f8ac2bfa5b4a6ed4f449f531bb37a0e3267\', \'another_et_another_et009.png\': \'18842b56f1b2eed1cbb148a4c8cb8d2c4e83a2b49d3d5f5029bc683798b12662\', \'et_et004.png\': \'8176eb231b3131ffa339a993eed009dffed9289105554245585a589460cf8130\'}, \'benchmark\': {\'benchmark_hash\': \'a78f0542413b8ac2ec8c262988ed5303c934490a2b31d2a45d3939dd6d3ab2a4\', \'split_hash\': \'170546743202f21b9bcc98376ff09a6eae2da16ce833115478a976c4f46b2057\', \'evaluator_hash\': \'c7ae94306ba0ae4ce9c00e9abc7e15bd99f3091fa063b5826b0dd567a6c3f261\', \'threshold_hash\': \'c6aec2fc2c8f9cb5f268f57333911f0835fcaba8aedc0eec67951ed729e68645\'}, \'weights_status\': \'Original versions unresolved\', \'weight_files\': {\'superpoint\': \'superpoint_v1.pth\', \'aliked\': \'aliked-n16.pth\', \'mast3r\': \'MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric.pth\', \'retrieval\': \'MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric_retrieval_trainingfree.pth\', \'codebook\': \'MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric_retrieval_codebook.pkl\', \'isc\': \'isc_ft_v107.pth.tar\', \'dinov2\': \'pytorch_model.bin\'}}\nprint(run(plan, Path(\'/kaggle/input\'), Path(\'/kaggle/working/imc_top10_reproduction_pilot.zip\')))\n')
try:
    subprocess.run([sys.executable,str(script)],check=True,timeout=1800)
except (subprocess.TimeoutExpired,subprocess.CalledProcessError) as exc:
    with zipfile.ZipFile(work/'imc_top10_reproduction_pilot.zip','w') as z:
        z.writestr('result.json',json.dumps(dict(status='BLOCKED',error=str(exc))))
    raise
